<a href="https://colab.research.google.com/github/guansong8/BA305-Straight-A-Students/blob/main/BA305.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Load the libraries used in this lab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Splitting the data, cross-validation, and the model itself
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Evaluation metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.metrics import roc_curve, roc_auc_score

In [ ]:
#look up csv file on my computer
!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

# fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# data (as pandas dataframes)
X = bank_marketing.data.features
y = bank_marketing.data.targets

# metadata
print(bank_marketing.metadata)

# variable information
print(bank_marketing.variables)


{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

In [ ]:
bank_df = pd.concat([X, y], axis=1)
bank_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


In [ ]:
bank_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  y            45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [ ]:
cols = ['duration','contact']
clean_bankdf = bank_df.drop(cols, axis=1)
clean_bankdf

,age,job,marital,education,default,balance,housing,loan,day_of_week,month,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,5,may,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,5,may,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,5,may,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,5,may,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,5,may,1,-1,0,NaN,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,17,nov,3,-1,0,NaN,yes
45207,71,retired,divorced,primary,no,1729,no,no,17,nov,2,-1,0,NaN,yes
45208,72,retired,married,secondary,no,5715,no,no,17,nov,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,17,nov,4,-1,0,NaN,no


In [ ]:
# Compare subscription rates based on previous contact --> whether previous contact matters
clean_bankdf["prior_contact"] = (
    clean_bankdf["previous"] > 0)

print(
    pd.crosstab(
        clean_bankdf["prior_contact"],
        clean_bankdf["y"],
        normalize="index"
    ) * 100
)

y                     no        yes
prior_contact                      
False          90.842669   9.157331
True           76.928667  23.071333


Customers who never contacted before had 9.16% subscription, who were contacted before had 23.07% subscription rate. There is a 13.91 percentage points difference.

In [ ]:
# difference between previous campaign outcomes
print(
    pd.crosstab(
        clean_bankdf["poutcome"],
        clean_bankdf["y"],
        normalize="index"
    ) * 100
)

y                no        yes
poutcome                      
failure   87.390329  12.609671
other     83.315217  16.684783
success   35.274653  64.725347


Customers with a previous successful campaign had a 64.73% subscription rate, which was 5.13 times as likely to subscribe as customers with a previous campaign failure.